## Fully Convolutional Networks (FCNs)

In [24]:
import torch 
import torch.nn.functional as F

In [25]:
import torchvision.models as models 
vgg=models.vgg16(pretrained=True)

c:\Users\moham\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\moham\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vgg = vgg.to(device) 

##### pool 3 
##### pool 4 
##### final output 

In [27]:
from torchsummary import summary
summary(vgg,(3,224,224),device=str(device))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 224, 224]           1,792
              ReLU-2         [-1, 64, 224, 224]               0
            Conv2d-3         [-1, 64, 224, 224]          36,928
              ReLU-4         [-1, 64, 224, 224]               0
         MaxPool2d-5         [-1, 64, 112, 112]               0
            Conv2d-6        [-1, 128, 112, 112]          73,856
              ReLU-7        [-1, 128, 112, 112]               0
            Conv2d-8        [-1, 128, 112, 112]         147,584
              ReLU-9        [-1, 128, 112, 112]               0
        MaxPool2d-10          [-1, 128, 56, 56]               0
           Conv2d-11          [-1, 256, 56, 56]         295,168
             ReLU-12          [-1, 256, 56, 56]               0
           Conv2d-13          [-1, 256, 56, 56]         590,080
             ReLU-14          [-1, 256,

In [28]:
## We will extract the output from  -- MaxPool2d-17          [-1, 256, 28, 28]                
## We will extract the output from  -- MaxPool2d-24          [-1, 512, 14, 14]  
## We will extract the output from  -- AdaptiveAvgPool2d-32            [-1, 512, 7, 7]              
import torch.nn as nn
class FCNB(nn.Module):
    def __init__(self,num_classes=2):
        super(FCNB,self).__init__()
        vgg=models.vgg16(pretrained=True)
        self.features=vgg.features #till convolotion and pooling layers 

        ## extract intermediate feature maps 
        self.pool3=self.features[:17]
        self.pool4=self.features[:24]
        self.pool5=self.features

        ## 1*1 convolutions to convert to class scorse 
        self.conv1x1_3=nn.Conv2d(256,num_classes,kernel_size=1)
        self.conv1x1_4=nn.Conv2d(512,num_classes,kernel_size=1)
        self.conv1x1_5=nn.Conv2d(512,num_classes,kernel_size=1)

        ## upsample the data 
        self.upsample32=nn.ConvTranspose2d(num_classes,num_classes,kernel_size=4,stride=2,padding=1)
        self.upsample16=nn.ConvTranspose2d(num_classes,num_classes,kernel_size=4,stride=2,padding=1)
        self.upsample8=nn.ConvTranspose2d(num_classes,num_classes,kernel_size=16,stride=8,padding=4)

    def forward(self,x):
        pool3_out=self.pool3(x)
        pool4_out=self.pool4(x)
        pool5_out=self.pool5(x)

        pool3_out=self.conv1x1_3(pool3_out)
        pool4_out=self.conv1x1_4(pool4_out)
        pool5_out=self.conv1x1_5(pool5_out)


        x=self.upsample32(pool5_out)+pool4_out
        x=self.upsample16(x)+pool3_out
        x=self.upsample8(x)

        return x
        















In [29]:
'''
I="INPUT SIZE" 
S="stride"
K="kernel_size"
P="Padding"
o=(I-1)*S+K-2*P
print(224/28)
print(224/7)
print(224/16)

input_size=224,224 
max_pool5=7,7==32
max_pool4=16,16==14
max_pool3=28,28==8

'''

'\nI="INPUT SIZE" \nS="stride"\nK="kernel_size"\nP="Padding"\no=(I-1)*S+K-2*P\nprint(224/28)\nprint(224/7)\nprint(224/16)\n\ninput_size=224,224 \nmax_pool5=7,7==32\nmax_pool4=16,16==14\nmax_pool3=28,28==8\n\n'

In [30]:
'''
print(224/28)
print(224/7)
print(224/16)
'''


'\nprint(224/28)\nprint(224/7)\nprint(224/16)\n'

In [32]:
model=FCNB(num_classes=1000)
dummy_input=torch.randn(1,3,224,224)
output=model(dummy_input)

print("Output shape :" ,output.shape)


Output shape : torch.Size([1, 1000, 224, 224])
